# Lab: Support Vector Machine (SVM)

## 1. Ý tưởng cốt lõi

Cho hai lớp điểm trong không gian, có vô số đường (siêu phẳng) tách được chúng. SVM hỏi: đường nào là tốt nhất?

Câu trả lời: đường tách *xa nhất* khỏi cả hai lớp. "Khoảng cách từ đường tách đến điểm gần nhất của mỗi lớp" gọi là **margin**. SVM tìm siêu phẳng có margin lớn nhất, còn gọi là **maximum margin classifier**.

Trực giác: margin lớn nghĩa là an toàn hơn khi gặp dữ liệu mới hơi lệch một chút.

## 2. Hard-margin SVM (dữ liệu tách tuyến tính)

Giả sử nhãn $y_i \in \{-1, +1\}$. Siêu phẳng là $w^T x + b = 0$. Ta muốn:
$$
y_i (w^T x_i + b) \ge 1, \quad \forall i
$$
Margin $= \frac{2}{\|w\|}$, nên tối đa hoá margin tương đương tối thiểu hoá $\frac{1}{2}\|w\|^2$.

**Bài toán tối ưu (primal):**
$$
\min_{w, b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w^T x_i + b) \ge 1
$$

## 3. Soft-margin SVM (dữ liệu không tách hoàn toàn)

Thực tế dữ liệu có nhiễu, có outlier. Cho phép một số điểm vi phạm margin bằng biến slack $\xi_i \ge 0$:
$$
y_i(w^T x_i + b) \ge 1 - \xi_i
$$

$$
\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C\sum_i \xi_i
$$

**Tham số $C$**:
- $C$ lớn: phạt vi phạm nặng, margin nhỏ, sát dữ liệu (dễ overfit).
- $C$ nhỏ: khoan dung hơn, margin lớn (bias cao hơn nhưng thường tổng quát hoá tốt hơn).

## 4. Bài toán đối ngẫu (dual) và kernel

Bằng nhân tử Lagrange, primal có thể chuyển sang **dual**:
$$
\max_{\lambda} \sum_i \lambda_i - \frac{1}{2}\sum_{i, j}\lambda_i \lambda_j y_i y_j (x_i \cdot x_j)
$$
$$
\text{s.t.} \quad 0 \le \lambda_i \le C, \quad \sum_i \lambda_i y_i = 0
$$

Điểm quan trọng: dual chỉ phụ thuộc vào *tích vô hướng* $x_i \cdot x_j$. Đây là cánh cửa đến **Kernel Trick**: thay $x_i \cdot x_j$ bằng một hàm $K(x_i, x_j)$ tương ứng với tích vô hướng trong một không gian cao chiều ngầm. SVM lúc đó học được ranh giới phi tuyến mà không cần thực sự nhảy lên không gian cao.

## 5. Các kernel phổ biến

- **Linear**: $K(x, y) = x \cdot y$.
- **Polynomial**: $K(x, y) = (x \cdot y + c)^d$.
- **RBF (Gaussian)**: $K(x, y) = \exp(-\gamma\|x - y\|^2)$. Phổ biến nhất, hoạt động tốt cho hầu hết dataset.

## 6. Support vectors

Sau khi train, chỉ những điểm có $\lambda_i > 0$ mới đóng góp vào quyết định. Đây là **support vectors**, thường chỉ một phần nhỏ của dữ liệu. Đó là lý do SVM gọi là *Support Vector* Machine.

---

## 7. Nhìn bằng hình: giải phẫu của margin

![Giải phẫu margin của SVM](images/01_hinh_hoc_margin.png)

*Siêu phẳng $w^Tx+b=0$ (nét liền đen) nằm chính giữa hai lề $w^Tx+b=\pm 1$ (nét đứt). Vector $w$ luôn vuông góc với siêu phẳng. Ba điểm được khoanh tròn xanh lá là support vector, chúng là các điểm quyết định vị trí của hai lề.*

### 7.1. Functional margin và geometric margin: hai thứ dễ nhầm

Với một điểm $(x_i, y_i)$, $y_i \in \{-1,+1\}$, ta định nghĩa:

| | Công thức | Ý nghĩa |
|---|---|---|
| **Functional margin** | $\hat{\gamma}_i = y_i\,(w^Tx_i + b)$ | Điểm số có dấu. Dương = phân loại đúng, càng lớn càng "tự tin". Không phải khoảng cách. |
| **Geometric margin** | $\gamma_i = \dfrac{y_i\,(w^Tx_i + b)}{\lVert w\rVert} = \dfrac{\hat{\gamma}_i}{\lVert w\rVert}$ | Khoảng cách hình học thật sự (đơn vị mét, cm...) từ điểm tới siêu phẳng. |

**Vì sao phải phân biệt?** Vì functional margin *có thể phóng to tuỳ ý mà không đổi mô hình*. Nhân cả $w$ và $b$ với $k = 1000$:

$$1000\,w^Tx + 1000\,b = 0 \iff w^Tx + b = 0$$

Vẫn đúng cùng một siêu phẳng, nhưng functional margin vừa tăng gấp 1000 lần. Nếu lấy functional margin làm mục tiêu tối đa hoá, bài toán sẽ vô nghĩa: cứ nhân $w$ lên là "tốt hơn" mãi mãi.

### 7.2. Vì sao chuẩn hoá được về $y_i(w^Tx_i+b) \ge 1$

Vì tỷ lệ $(w,b) \to (kw, kb)$ là tự do, ta cố định nó lại bằng một quy ước: chọn $k$ sao cho điểm gần siêu phẳng nhất có functional margin đúng bằng 1:

$$\min_i\; y_i(w^Tx_i + b) = 1 \quad\Longrightarrow\quad y_i(w^Tx_i + b) \ge 1 \;\; \forall i$$

Đây không phải một giả thiết thêm vào, mà chỉ là cách chọn đơn vị đo, giống như quy ước đo bằng mét thay vì centimet. Khi đã cố định như vậy:

$$\text{geometric margin của điểm gần nhất} = \frac{1}{\|w\|}, \qquad \text{độ rộng dải margin} = \frac{2}{\|w\|}$$

$$\max_{w,b} \frac{2}{\|w\|} \;\;\iff\;\; \min_{w,b} \|w\| \;\;\iff\;\; \min_{w,b} \frac{1}{2}\|w\|^2$$

Bình phương và hệ số $\tfrac{1}{2}$ chỉ để đạo hàm cho đẹp ($\nabla = w$) và để bài toán trở thành **quy hoạch toàn phương lồi**, luôn có nghiệm toàn cục duy nhất, không kẹt ở cực tiểu địa phương như mạng nơ-ron.


## 8. Vì sao margin lớn lại tổng quát hoá tốt hơn?

![Vì sao margin lớn tốt hơn](images/02_vi_sao_margin_lon.png)

*Trái: cả ba đường A, B, C đều phân loại đúng 100% dữ liệu train, nên train accuracy không giúp ta chọn. Phải: một điểm test mới rơi vào khe giữa đường C và siêu phẳng SVM; đường margin hẹp đoán sai, đường max-margin đoán đúng.*

Trực giác: margin là vùng đệm. Dữ liệu thật luôn có nhiễu đo đạc; một điểm mới thường lệch khỏi vị trí "lý tưởng" của nó vài phần trăm. Nếu ranh giới nằm sát một lớp, chỉ cần lệch một chút là nhãn bị đổi. Margin rộng nghĩa là dung sai lớn.

Về mặt lý thuyết (không cần thuộc, chỉ cần biết là có): lý thuyết học thống kê chứng minh được rằng với tập siêu phẳng có geometric margin ít nhất $\gamma$ trên dữ liệu nằm gọn trong quả cầu bán kính $R$, chiều VC bị chặn bởi

$$\text{VC} \le \left\lceil \frac{R^2}{\gamma^2} \right\rceil$$

Điều đặc biệt: cận này không chứa số chiều $d$. Đó là lý do SVM vẫn hoạt động tốt khi số feature rất lớn (thậm chí vô hạn, như với RBF kernel), trong khi nhiều mô hình khác gặp khó khăn vì lời nguyền số chiều.

Lưu ý rằng margin lớn chỉ tốt khi dữ liệu tương đối sạch. Nếu có outlier, ép phân loại đúng toàn bộ tập train sẽ làm hỏng margin, như mục sau sẽ cho thấy. Đó là lý do cần soft-margin.


## 9. Hard-margin, soft-margin và biến slack $\xi_i$

![Hard-margin vs soft-margin](images/03_hard_vs_soft_margin.png)

*Chỉ thêm một điểm outlier (ngôi sao đỏ): hard-margin buộc phải chiều theo nó nên margin co từ 1.60 xuống gần 0, mô hình gần như vô dụng. Soft-margin với $C=0.1$ chấp nhận outlier vi phạm (mũi tên cam là $\xi_i$) và giữ được margin 2.87.*

Hard-margin có hai nhược điểm lớn:

1. Vô nghiệm nếu dữ liệu không tách được tuyến tính (chỉ cần 1 điểm dán nhầm nhãn là bài toán không còn miền khả thi).
2. Rất nhạy với outlier, như hình giữa.

Soft-margin nới lỏng ràng buộc bằng biến slack $\xi_i \ge 0$:

$$\min_{w,b,\xi} \;\; \frac{1}{2}\|w\|^2 + C\sum_{i=1}^{n}\xi_i \qquad \text{s.t.}\quad y_i(w^Tx_i+b) \ge 1 - \xi_i,\;\; \xi_i \ge 0$$

Có thể đọc giá trị $\xi_i$ như một mức phạt:

| Giá trị $\xi_i$ | Điểm nằm ở đâu | Phân loại đúng? |
|---|---|---|
| $\xi_i = 0$ | Ngoài lề hoặc đúng trên lề, vùng an toàn | Đúng |
| $0 < \xi_i < 1$ | Lấn vào bên trong dải margin nhưng vẫn đúng phía | Đúng |
| $\xi_i = 1$ | Nằm đúng trên siêu phẳng | Ranh giới (50/50) |
| $\xi_i > 1$ | Vượt sang phía bên kia | Sai |

Vì $\sum_i \xi_i$ là cận trên của số điểm bị phân loại sai, hàm mục tiêu $\frac{1}{2}\|w\|^2 + C\sum\xi_i$ đọc thành lời là: *"margin rộng nhất có thể, đồng thời tổng mức vi phạm nhỏ nhất có thể, và $C$ là tỷ giá quy đổi giữa hai mong muốn đó."*


## 10. Điều kiện KKT: vì sao SVM chỉ cần nhớ vài điểm

Chuyển primal sang dual bằng nhân tử Lagrange $\lambda_i \ge 0$. Tại nghiệm tối ưu, các **điều kiện KKT** (Karush-Kuhn-Tucker) phải thoả:

$$\textbf{(1) Dừng: }\quad w = \sum_{i=1}^{n}\lambda_i y_i x_i, \qquad \sum_{i=1}^{n}\lambda_i y_i = 0$$

$$\textbf{(2) Bù trừ (complementary slackness): }\quad \lambda_i\big[\,y_i(w^Tx_i+b) - 1 + \xi_i\,\big] = 0$$

$$\textbf{(3) Ràng buộc hộp: }\quad 0 \le \lambda_i \le C, \qquad (C - \lambda_i)\,\xi_i = 0$$

Điều kiện (2) là chìa khoá. Nó nói: với mỗi điểm, hoặc $\lambda_i = 0$, hoặc ràng buộc phải chặt (điểm nằm đúng trên lề). Từ đó dữ liệu chia làm đúng ba nhóm:

| Nhóm | $\lambda_i$ | Vị trí | Vai trò |
|---|---|---|---|
| Điểm "nhàn rỗi" | $\lambda_i = 0$ | Nằm ngoài lề, an toàn | Không đóng góp gì. Xoá đi vẫn ra đúng nghiệm cũ. |
| Support vector trên lề | $0 < \lambda_i < C$ | Nằm đúng trên lề, $\xi_i = 0$ | Xác định siêu phẳng; dùng để tính $b$ |
| Support vector vi phạm | $\lambda_i = C$ | Trong dải margin hoặc sai phía, $\xi_i > 0$ | Là các điểm "khó", bị phạt |

Vì $w = \sum_i \lambda_i y_i x_i$ và đa số $\lambda_i = 0$, ta có:

$$w = \sum_{i \in SV} \lambda_i y_i x_i, \qquad f(x) = \sum_{i \in SV} \lambda_i y_i\, K(x_i, x) + b$$

Ba hệ quả thực tế:

1. Mô hình thưa (sparse). Chỉ support vector được lưu lại (`clf.support_vectors_`). Dataset 10.000 mẫu có thể chỉ cần nhớ 300 điểm.
2. Xoá dữ liệu không phải SV, nghiệm không đổi. Có thể tự kiểm tra: bỏ hết các điểm ở xa rồi train lại, kết quả vẫn ra cùng một $w, b$.
3. Chi phí predict tỷ lệ với số SV, không phải số mẫu train. Nếu số SV ≈ số mẫu (thường do $\gamma$ hoặc $C$ đặt sai) thì mô hình vừa chậm vừa đang overfit. Đó là một tín hiệu chẩn đoán rất hữu ích.


## 11. Đọc SVM dưới lăng kính hàm mất mát: hinge loss

![Hinge loss và các hàm mất mát khác](images/08_hinge_loss_vs_cac_loss_khac.png)

*Trái: hinge loss (đỏ) luôn nằm trên 0-1 loss (đen) và là hàm lồi, nên tối thiểu hinge sẽ kéo 0-1 loss xuống theo, mà lại tối ưu được bằng thuật toán. Phải: hinge tắt hẳn về 0 khi $m \ge 1$ (mô hình thưa), còn log loss của Logistic Regression thì không bao giờ bằng 0.*

Ràng buộc $y_i(w^Tx_i+b) \ge 1-\xi_i$ với $\xi_i \ge 0$ và $\xi_i$ nhỏ nhất có thể tương đương với việc chọn thẳng:

$$\xi_i = \max\big(0,\; 1 - y_i(w^Tx_i+b)\big)$$

Thay vào, bài toán SVM trở thành một bài tối ưu không ràng buộc:

$$\boxed{\;\min_{w,b}\;\; \frac{1}{2}\|w\|^2 \;+\; C\sum_{i=1}^{n} \max\big(0,\;1 - y_i(w^Tx_i+b)\big)\;}$$

Chia hai vế cho $C$ và đặt $\lambda = \dfrac{1}{2C}$:

$$\min_{w,b}\;\; \underbrace{\sum_i \max(0, 1 - y_i f(x_i))}_{\text{mất mát (fit dữ liệu)}} \;+\; \underbrace{\lambda\|w\|^2}_{\text{regularization L2}}$$

Đây chính xác là khuôn "loss + L2 penalty" của Ridge/Logistic Regression đã học ở Lab 01 và 02. Kết luận quan trọng:

$$C \;\leftrightarrow\; \frac{1}{\lambda}: \qquad C \text{ lớn } = \lambda \text{ nhỏ } = \text{ ít regularization } = \text{ dễ overfit}$$

### SVM và Logistic Regression khác nhau ở đúng một chỗ

| | SVM (hinge) | Logistic Regression (log loss) |
|---|---|---|
| Mất mát | $\max(0, 1-m)$ | $\log(1+e^{-m})$ |
| Bằng 0 khi? | Bằng 0 hẳn khi $m \ge 1$ | Không bao giờ bằng 0 |
| Điểm nào ảnh hưởng nghiệm? | Chỉ support vector (thưa) | Mọi điểm, kể cả điểm rất xa (không thưa) |
| Đầu ra tự nhiên | Điểm số có dấu (khoảng cách) | Xác suất hợp lệ |
| Khả vi? | Không tại $m=1$, dùng subgradient | Khả vi mọi nơi |
| Nhạy outlier | Vừa phải (loss tăng tuyến tính) | Vừa phải (loss tăng tuyến tính khi $m$ rất âm) |

Hai mô hình cho kết quả rất giống nhau trên phần lớn dataset. Chọn SVM khi cần biên rõ ràng và mô hình thưa; chọn Logistic Regression khi cần xác suất.

`LinearSVC` của sklearn mặc định dùng squared hinge $\max(0,1-m)^2$ (đường cam đứt nét trong hình) vì nó khả vi, tối ưu nhanh hơn, nhưng phạt điểm vi phạm nặng hơn nên nhạy outlier hơn. Đổi bằng `loss='hinge'` nếu cần đúng chuẩn SVM.


## 12. Kernel dưới góc nhìn toán học

![Kernel trick nâng chiều](images/05_kernel_trick_nang_chieu_3d.png)

*Hình quan trọng nhất của bài. Trái: hai vòng tròn đồng tâm, không đường thẳng nào tách nổi. Phải: sau khi thêm chiều thứ ba $x_3 = x_1^2 + x_2^2$, một mặt phẳng nằm ngang tách được hoàn hảo. Cắt mặt phẳng đó trở lại 2D, ta được đúng đường tròn ở hình trái.*

### 12.1. Định nghĩa

Một hàm $K$ là **kernel hợp lệ** nếu tồn tại một ánh xạ $\phi$ sang không gian đặc trưng $\mathcal{H}$ sao cho:

$$K(x, x') = \langle \phi(x),\, \phi(x') \rangle$$

Một ví dụ cụ thể. Với $x = (x_1, x_2) \in \mathbb{R}^2$ và $K(x,x') = (x^Tx')^2$:

$$(x^Tx')^2 = (x_1x_1' + x_2x_2')^2 = x_1^2x_1'^2 + 2x_1x_2x_1'x_2' + x_2^2x_2'^2 = \langle \phi(x), \phi(x')\rangle$$

với $\phi(x) = (x_1^2,\; \sqrt{2}\,x_1x_2,\; x_2^2)$. Tính vế trái: 2 phép nhân và 1 phép bình phương. Tính vế phải: dựng 2 vector 3 chiều rồi nhân vô hướng. Với đa thức bậc $d$ trong không gian $n$ chiều, $\phi$ có $\binom{n+d-1}{d}$ thành phần (tăng rất nhanh theo tổ hợp), trong khi $K$ vẫn chỉ là *một phép nhân vô hướng rồi luỹ thừa*. Đó chính là "trick".

### 12.2. Điều kiện Mercer và ma trận Gram

Định lý Mercer: $K$ đối xứng và liên tục là kernel hợp lệ khi và chỉ khi với mọi tập điểm hữu hạn $\{x_1,\dots,x_n\}$, **ma trận Gram**

$$\mathbf{K} = \begin{pmatrix} K(x_1,x_1) & \cdots & K(x_1,x_n) \\ \vdots & \ddots & \vdots \\ K(x_n,x_1) & \cdots & K(x_n,x_n)\end{pmatrix}$$

là **nửa xác định dương** (positive semi-definite): $c^T\mathbf{K}\,c \ge 0\;\;\forall c \in \mathbb{R}^n$, tương đương mọi trị riêng $\ge 0$.

Vì sao cần điều kiện này? Vì nó đúng là thứ bảo đảm bài toán dual vẫn lồi. Nếu $\mathbf{K}$ không PSD, hàm mục tiêu dual không còn lõm, nghiệm có thể không tồn tại hoặc solver không hội tụ.

Tính chất đóng (tiện khi tự xây kernel): nếu $K_1, K_2$ là kernel hợp lệ thì $K_1+K_2$, $aK_1\,(a>0)$, $K_1 \cdot K_2$, và $f(x)K_1(x,x')f(x')$ cũng là kernel hợp lệ.

### 12.3. Bảng các kernel thông dụng

| Kernel | Công thức | Tham số | Không gian $\phi$ | Khi nào dùng |
|---|---|---|---|---|
| **Linear** | $K = x^Tx'$ | không | Chính không gian gốc | $d$ lớn & thưa (văn bản, TF-IDF, gen); $n$ rất lớn; cần diễn giải hệ số |
| **Polynomial** | $K = (\gamma\,x^Tx' + r)^d$ | $\gamma, r, d$ | Hữu hạn, $\binom{n+d-1}{d}$ chiều | Khi tương tác bậc thấp giữa feature là quan trọng (thị giác cổ điển, NLP n-gram) |
| **RBF (Gauss)** | $K = \exp(-\gamma\|x-x'\|^2)$ | $\gamma$ | Vô hạn chiều | Nên thử trước. Dữ liệu số, $d$ vừa phải, không rõ hình dạng ranh giới |
| **Sigmoid** | $K = \tanh(\gamma\,x^Tx' + r)$ | $\gamma, r$ | Không xác định rõ (chỉ PSD với một số $\gamma, r$) | Hiếm khi thắng RBF. Chủ yếu tồn tại vì lý do lịch sử (giống 1 tầng nơ-ron) |

![So sánh các kernel](images/07_so_sanh_cac_kernel.png)

*Cùng một dữ liệu `make_moons`, mỗi kernel là một "giả thuyết" khác nhau về hình dạng ranh giới. Chú ý sigmoid cho kết quả tệ nhất (72.3%), đúng như cảnh báo ở bảng trên.*

Lưu ý: kernel poly và sigmoid rất nhạy với scale của feature (vì có $x^Tx'$ nâng luỹ thừa/đưa vào tanh, dễ tràn số hoặc bão hoà). Hình trên đã chuẩn hoá dữ liệu trước; nếu không, poly bậc 3 thường cho ranh giới vô nghĩa.


## 13. Hiểu $\gamma$ của RBF và cách tune cặp $(C, \gamma)$

![RBF kernel như độ giống nhau theo khoảng cách](images/09_rbf_kernel_do_giong_nhau.png)

*RBF kernel thực chất là một thước đo độ giống nhau: hai điểm trùng nhau cho $K=1$, càng xa nhau $K$ càng tụt về 0. $\gamma$ quyết định tụt nhanh hay chậm, tức là bán kính ảnh hưởng của mỗi support vector.*

Viết lại RBF theo dạng Gauss quen thuộc:

$$K(x,x') = \exp\!\left(-\gamma\|x-x'\|^2\right) = \exp\!\left(-\frac{\|x-x'\|^2}{2\sigma^2}\right), \qquad \gamma = \frac{1}{2\sigma^2}$$

Nên bề rộng ảnh hưởng $\sigma \propto 1/\sqrt{\gamma}$.

| $\gamma$ | Bề rộng ảnh hưởng | Ranh giới | Rủi ro |
|---|---|---|---|
| Rất nhỏ (0.001) | Rất rộng, mọi điểm "giống" mọi điểm | Gần như thẳng | Underfit: RBF thoái hoá thành gần tuyến tính |
| Vừa | Cỡ khoảng cách trung bình giữa các điểm | Cong hợp lý | Tốt |
| Rất lớn (100) | Rất hẹp, mỗi điểm là một ốc đảo | Bao quanh từng điểm train | Overfit: train acc gần 100%, test acc thấp; số SV gần bằng số mẫu |

### `gamma='scale'` trong sklearn là gì?

Mặc định của sklearn từ phiên bản 0.22:

$$\texttt{gamma='scale'} \;=\; \frac{1}{d \cdot \mathrm{Var}(X)}, \qquad \texttt{gamma='auto'} \;=\; \frac{1}{d}$$

với $d$ = số feature và $\mathrm{Var}(X)$ = phương sai của toàn bộ ma trận $X$ (tính trên tất cả phần tử). Ý tưởng: chuẩn hoá $\gamma$ theo độ "trải" của dữ liệu để giá trị mặc định hợp lý bất kể đơn vị đo. Lưu ý: nếu bạn đã `StandardScaler` (mỗi cột có phương sai 1) thì $\mathrm{Var}(X) \approx 1$ và `'scale'` $\approx 1/d$, hai lựa chọn trùng nhau.

### Tune $(C, \gamma)$ theo lưới log

Đây là cặp tham số tương tác mạnh với nhau: $\gamma$ lớn (mô hình linh hoạt) cần $C$ nhỏ (phạt nhẹ) để không overfit, và ngược lại. Vì thế không nên tune riêng từng cái mà luôn dùng lưới 2 chiều.

1. Scale dữ liệu trước (`StandardScaler`), cần thiết vì $\|x-x'\|^2$ phụ thuộc đơn vị đo.
2. Lưới thô, bước nhân 10: `C ∈ {0.01, 0.1, 1, 10, 100, 1000}`, `gamma ∈ {1e-4, 1e-3, 1e-2, 1e-1, 1, 10}`.
3. Nhìn heatmap CV score. Vùng tốt thường là một dải chéo (C lớn đi kèm gamma nhỏ).
4. Lưới tinh quanh điểm tốt nhất, bước nhân 2 đến 3.
5. Kiểm tra `len(clf.support_vectors_) / n`. Trên 60 đến 70% là dấu hiệu $\gamma$ quá lớn.

```python
param_grid = {'C': np.logspace(-2, 3, 6), 'gamma': np.logspace(-4, 1, 6)}
GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, n_jobs=-1)
```


### Ảnh động: gamma quét từ 0.05 lên 200

![Ảnh động cho thấy ranh giới RBF thay đổi khi gamma tăng dần](images/anim_gamma_rbf.gif)

*Cùng một bộ `make_moons`, cùng $C=1$, chỉ có $\gamma$ chạy theo thang log từ 0.05 lên 200. Ở đầu đoạn phim ranh giới gần như một đường thẳng và hai con số train accuracy, test accuracy bám sát nhau. Càng về sau ranh giới càng co lại thành những ốc đảo quanh từng điểm train, số support vector tăng vọt, train accuracy leo lên gần 100% còn test accuracy thì tụt xuống. Khoảnh khắc hai con số đó tách rời nhau chính là lúc mô hình bắt đầu overfit.*


# THỰC HÀNH 1: Linear SVM trên dữ liệu 2D giả lập

Sinh hai cụm có thể tách tuyến tính, train SVM, vẽ siêu phẳng + margin + support vectors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

np.random.seed(42)

In [ ]:
# Hai cụm tách tuyến tính
X, y = make_blobs(n_samples=80, centers=2, cluster_std=1.0, random_state=42)

clf = SVC(kernel='linear', C=1.0)
clf.fit(X, y)

def plot_svm(clf, X, y, title):
    plt.figure(figsize=(7, 6))
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30, edgecolor='k')

    # Vẽ ranh giới + margin
    xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200)
    yy = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200)
    XX, YY = np.meshgrid(xx, yy)
    grid = np.c_[XX.ravel(), YY.ravel()]
    Z = clf.decision_function(grid).reshape(XX.shape)
    plt.contour(XX, YY, Z, levels=[-1, 0, 1], colors=['red', 'black', 'red'],
                linestyles=['--', '-', '--'])

    # Tô đậm support vectors
    plt.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                s=200, facecolors='none', edgecolors='lime', linewidths=2,
                label=f'{len(clf.support_vectors_)} support vectors')
    plt.legend(); plt.title(title); plt.grid(alpha=0.3)
    plt.show()

plot_svm(clf, X, y, f'Linear SVM, C={clf.C}')
print(f'w = {clf.coef_[0]},  b = {clf.intercept_[0]:.3f}')
print(f'Margin width = 2/||w|| = {2 / np.linalg.norm(clf.coef_):.3f}')

### Ảnh hưởng của tham số C

Thêm vài điểm "khó" gần boundary để thấy C ảnh hưởng thế nào.

In [ ]:
# Dữ liệu hơi bị overlap
X2, y2 = make_blobs(n_samples=80, centers=2, cluster_std=1.8, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, c in zip(axes, [0.01, 1, 100]):
    clf = SVC(kernel='linear', C=c).fit(X2, y2)
    ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='coolwarm', s=30, edgecolor='k')
    xx = np.linspace(X2[:, 0].min()-1, X2[:, 0].max()+1, 100)
    yy = np.linspace(X2[:, 1].min()-1, X2[:, 1].max()+1, 100)
    XX, YY = np.meshgrid(xx, yy)
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contour(XX, YY, Z, levels=[-1, 0, 1], colors=['red', 'black', 'red'],
               linestyles=['--', '-', '--'])
    ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
               s=150, facecolors='none', edgecolors='lime', linewidths=2)
    ax.set_title(f'C = {c}, {len(clf.support_vectors_)} SVs, margin = {2/np.linalg.norm(clf.coef_):.2f}')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Nhìn kỹ hơn: $C$ đổi cả margin lẫn số support vector

![Ảnh hưởng của tham số C](images/04_anh_huong_tham_so_C.png)

*Cùng một dữ liệu có chồng lấn. $C=0.01$: margin 4.52 và 39 support vector, mô hình khoan dung, dựa vào toàn cục. $C=100$: margin 1.10, chỉ 11 SV, mô hình chỉ dựa vào vài điểm sát biên, dễ bị một outlier kéo lệch.*

| | $C$ nhỏ | $C$ lớn |
|---|---|---|
| Regularization ($\lambda = 1/2C$) | Mạnh | Yếu |
| Độ rộng margin | Rộng | Hẹp |
| Số support vector | Nhiều | Ít |
| Bias / Variance | Bias cao, variance thấp | Bias thấp, variance cao |
| Train accuracy | Thấp hơn | Cao hơn (có thể 100%) |
| Rủi ro | Underfit | Overfit |

Một cách chẩn đoán nhanh: in `len(clf.support_vectors_)` sau mỗi lần train. Số SV gần bằng số mẫu cho thấy $C$ quá nhỏ hoặc $\gamma$ quá lớn; số SV chỉ còn 2 hoặc 3 trên dữ liệu nhiễu cho thấy $C$ quá lớn.


# THỰC HÀNH 2: Kernel SVM trên dữ liệu phi tuyến

Dùng `make_moons`: hai trăng lưỡi liềm cài vào nhau, không tách được tuyến tính.

In [ ]:
X, y = make_moons(n_samples=200, noise=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    clf = SVC(kernel=kernel, C=1.0, gamma='scale', degree=3).fit(X, y)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=30, edgecolor='k')
    xx = np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 200)
    yy = np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 200)
    XX, YY = np.meshgrid(xx, yy)
    Z = clf.decision_function(np.c_[XX.ravel(), YY.ravel()]).reshape(XX.shape)
    ax.contourf(XX, YY, Z, levels=20, cmap='coolwarm', alpha=0.3)
    ax.contour(XX, YY, Z, levels=[0], colors='black', linewidths=2)
    ax.set_title(f'Kernel: {kernel}, train acc = {clf.score(X, y)*100:.1f}%')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Kernel linear dừng ở khoảng 85% trên dữ liệu cong, còn kernel RBF đạt trên 95% nhờ học được ranh giới cong.


### Nhìn kỹ hơn: $\gamma$ biến ranh giới từ mượt thành "ốc đảo"

![Ảnh hưởng của gamma trong RBF](images/06_anh_huong_gamma_rbf.png)

*Cùng `make_moons`, cùng $C=1$, chỉ đổi $\gamma$. Từ trái sang phải: $\gamma=0.1$ gần như một đường thẳng (underfit); $\gamma=1$ bám đúng hình lưỡi liềm; $\gamma=10$ bắt đầu uốn theo nhiễu; $\gamma=100$ vẽ một "ốc đảo" quanh từng điểm train, train acc 98.2% nhưng 202/220 mẫu trở thành support vector, dấu hiệu overfit kinh điển.*

Ba con số nên in ra mỗi lần thử $\gamma$: train accuracy, CV accuracy và số support vector. Khi train acc tăng mà CV acc giảm và số SV tăng mạnh thì $\gamma$ đã quá lớn.


## 14. SVM trên dataset thật (Iris)

In [ ]:
from sklearn.datasets import load_iris

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# SVM nhạy với scale nên chuẩn hoá trước; fit scaler chỉ trên train để tránh rò rỉ
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Grid search tìm best C, gamma
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.01, 0.1, 1]}
gs = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
gs.fit(X_train_s, y_train)

print(f'Best params: {gs.best_params_}')
print(f'Best CV acc: {gs.best_score_*100:.2f}%')
print(f'Test acc:    {gs.score(X_test_s, y_test)*100:.2f}%')
print()
print(classification_report(y_test, gs.predict(X_test_s),
                            target_names=data.target_names))

## 15. Đa lớp, độ phức tạp và SVM cho dữ liệu lớn

### 15.1. SVM vốn là bài toán nhị phân, làm sao chạy được Iris 3 lớp?

SVM chỉ biết vẽ một siêu phẳng giữa hai lớp. Với $K$ lớp, sklearn ghép nhiều bộ nhị phân lại:

| Chiến lược | Số bộ phân loại | Ai dùng | Cách quyết định | Ưu / nhược |
|---|---|---|---|---|
| **One-vs-One (OvO)** | $\dfrac{K(K-1)}{2}$ | `SVC`, `NuSVC` | Bỏ phiếu: lớp nào thắng nhiều trận nhất | Mỗi bộ chỉ train trên dữ liệu của 2 lớp, nên mỗi bài nhỏ và cân bằng. Nhưng số bộ tăng bậc hai theo $K$ |
| **One-vs-Rest (OvR)** | $K$ | `LinearSVC`, `LogisticRegression` | Lấy lớp có `decision_function` lớn nhất | Ít bộ hơn, dễ diễn giải. Nhưng mỗi bộ train trên toàn bộ dữ liệu và bị mất cân bằng ($1$ chọi $K-1$) |

Với Iris ($K=3$): OvO tạo $3$ bộ (setosa/versicolor, setosa/virginica, versicolor/virginica). Với $K=10$: OvO cần 45 bộ, OvR chỉ 10 bộ.

Vì sao `SVC` chọn OvO dù số bộ nhiều hơn? Vì độ phức tạp huấn luyện SVM tăng nhanh hơn tuyến tính theo số mẫu (khoảng $n^2$ đến $n^3$). Chia $n$ mẫu thành các cặp lớp nhỏ rồi train nhiều lần thường rẻ hơn train $K$ lần trên toàn bộ $n$ mẫu.

`SVC(decision_function_shape='ovr')` chỉ định dạng lại đầu ra thành $K$ cột cho tiện dùng; bên trong vẫn huấn luyện theo OvO.

### 15.2. Độ phức tạp: vì sao SVM sợ dữ liệu lớn

| | Chi phí |
|---|---|
| Huấn luyện (kernel SVM, libsvm) | Giữa $O(n^2 d)$ và $O(n^3 d)$ |
| Bộ nhớ (cache ma trận kernel) | Tới $O(n^2)$, với $n=100{,}000$ là ~40 GB nếu cache đầy |
| Dự đoán 1 mẫu | $O(n_{SV} \cdot d)$, tỷ lệ với số support vector chứ không phải số mẫu |

Kinh nghiệm chung: $n \lesssim 10^4$ thì `SVC` thoải mái; $n \sim 10^5$ đã rất chậm; $n > 10^6$ thì không khả thi.

Thay thế khi dữ liệu lớn:

| Công cụ | Độ phức tạp | Ghi chú |
|---|---|---|
| `LinearSVC` (liblinear) | ~$O(nd)$ | Chỉ kernel tuyến tính, nhưng rất nhanh. Mặc định squared hinge + OvR |
| `SGDClassifier(loss='hinge')` | $O(nd)$ mỗi epoch, online | Chính là SVM tuyến tính huấn luyện bằng SGD. Chạy được dữ liệu không vừa RAM (`partial_fit`) |
| `Nystroem` / `RBFSampler` + `LinearSVC` | ~$O(nmd)$ | Xấp xỉ kernel RBF bằng $m$ chiều đặc trưng rồi dùng model tuyến tính, gần được chất lượng RBF với tốc độ tuyến tính |


## 16. SVR: SVM cho bài toán hồi quy

![SVR và ống epsilon](images/10_svr_ong_epsilon.png)

*Ý tưởng lật ngược: thay vì muốn dữ liệu nằm ngoài lề, SVR muốn dữ liệu nằm trong một ống rộng $\pm\epsilon$ quanh đường dự đoán. Điểm nằm trong ống (xanh) có mất mát bằng 0; chỉ điểm nằm ngoài (đỏ) mới bị phạt và trở thành support vector.*

Hàm mất mát **$\epsilon$-insensitive**:

$$L_\epsilon(y, \hat{y}) = \max\big(0,\; |y - \hat{y}| - \epsilon\big)$$

$$\min_{w,b,\xi,\xi^*} \; \frac{1}{2}\|w\|^2 + C\sum_i (\xi_i + \xi_i^*) \quad \text{s.t.}\quad |y_i - f(x_i)| \le \epsilon + \xi_i^{(*)}$$

| Tham số | Ý nghĩa | Đặt sai thì sao |
|---|---|---|
| $\epsilon$ | Bán kính ống, mức sai lệch được "tha bổng" | $\epsilon$ quá lớn: mô hình quá đơn giản, thành đường gần như phẳng. Quá nhỏ: gần như mọi điểm là SV, mất tính thưa |
| $C$ | Phạt cho phần vượt ống | Như SVM: lớn = bám dữ liệu, nhỏ = mượt |
| $\gamma$ | Như RBF thường | Như trên |

So với hồi quy tuyến tính dùng MSE, SVR có hai khác biệt lớn: (1) sai số nhỏ hơn $\epsilon$ hoàn toàn không bị phạt, nên mô hình thưa và ít nhạy với nhiễu nhỏ; (2) phần vượt ống bị phạt tuyến tính chứ không bình phương, nên ít bị outlier kéo lệch hơn MSE (xem lại Lab 01, phần Huber).

Nhớ scale cả `y` khi dùng SVR: $\epsilon$ tính theo đơn vị của $y$. Nếu $y$ là giá nhà tính bằng đồng, $\epsilon = 0.1$ (mặc định) là vô nghĩa. Dùng `TransformedTargetRegressor` hoặc scale thủ công.


## 17. Vì sao SVM không cho xác suất một cách tự nhiên

`decision_function(x)` trả về $f(x) = w^Tx + b$, một **khoảng cách có dấu** tới siêu phẳng, giá trị chạy từ $-\infty$ đến $+\infty$. Nó không phải xác suất và cũng không được huấn luyện để trở thành xác suất: hàm mục tiêu hinge chỉ quan tâm "vượt qua mốc $m \ge 1$ hay chưa", không quan tâm điểm nằm xa bao nhiêu sau mốc đó.

### Platt scaling: cách vá

Gắn thêm một hàm sigmoid lên đầu ra rồi fit hai tham số $A, B$ bằng maximum likelihood:

$$P(y = 1 \mid x) = \frac{1}{1 + \exp\big(A \cdot f(x) + B\big)}$$

Trong sklearn, `SVC(probability=True)` làm đúng việc này. Ba điều cần biết:

1. Tốn kém. Để tránh overfit, $A, B$ được fit bằng cross-validation 5-fold nội bộ, nên sklearn phải huấn luyện SVM thêm 5 lần nữa. Thời gian `fit` tăng khoảng 5 đến 6 lần.
2. `predict_proba` có thể mâu thuẫn với `decision_function`. Vì sigmoid được fit trên các fold CV chứ không phải trên mô hình cuối, có trường hợp `np.argmax(predict_proba(x))` khác với `predict(x)` (vốn dựa trên `decision_function`). Đây là hành vi đã được ghi rõ trong tài liệu sklearn, không phải bug.
3. Với dataset nhỏ (dưới khoảng 100 mẫu), Platt scaling dễ tự overfit, và xác suất còn tệ hơn khi không hiệu chỉnh.

### Khuyến nghị

| Bạn cần | Nên làm |
|---|---|
| Chỉ cần nhãn | Dùng `decision_function` / `predict`. Không bật `probability=True` |
| Cần xếp hạng (ROC-AUC, top-k) | Dùng thẳng `decision_function`, vì AUC chỉ quan tâm thứ tự chứ không cần xác suất |
| Cần xác suất thật (định giá rủi ro, chi phí quyết định) | Dùng `CalibratedClassifierCV(SVC(), method='sigmoid' hoặc 'isotonic', cv=5)` và kiểm tra bằng calibration curve và Brier score (xem Lab 07) |
| Cần xác suất mà dữ liệu lớn | Cân nhắc `LogisticRegression` ngay từ đầu, mô hình này cho xác suất một cách tự nhiên |


## 18. Bảng bẫy thường gặp khi dùng SVM

| # | Bẫy | Vì sao sai | Cách sửa |
|---|---|---|---|
| 1 | Quên scale feature | RBF/poly dựa trên $\lVert x-x'\rVert^2$ và $x^Tx'$. Một cột tính bằng đồng (10⁶) sẽ lấn át cột tính bằng năm (10¹), mô hình chỉ còn nhìn thấy một feature | `make_pipeline(StandardScaler(), SVC())`, luôn luôn |
| 2 | Scale trước khi split | Rò rỉ dữ liệu (data leakage): thống kê của test đã lọt vào train | Dùng `Pipeline` trong `GridSearchCV`; scaler chỉ `fit` trên train |
| 3 | Tune $C$ và $\gamma$ riêng lẻ | Hai tham số tương tác mạnh; tối ưu từng cái cho ra điểm rất xa tối ưu chung | Luôn dùng lưới 2 chiều (`GridSearchCV` với cả hai) |
| 4 | Dùng RBF cho dữ liệu rất cao chiều và thưa (TF-IDF, one-hot, gen) | Ở chiều cao, mọi khoảng cách $\lVert x-x'\rVert$ gần bằng nhau, nên $K$ gần như hằng số và RBF mất tác dụng; lại tốn thời gian gấp bội | Dùng `kernel='linear'` hoặc `LinearSVC`. Với $d > n$, kernel tuyến tính gần như luôn đủ |
| 5 | Bật `probability=True` khi không cần | Chậm gấp ~5 lần, và `predict_proba` có thể mâu thuẫn với `predict` | Chỉ bật khi thật sự cần xác suất; nếu chỉ cần AUC thì dùng `decision_function` |
| 6 | Đưa dataset 500k mẫu vào `SVC` | $O(n^2)$ tới $O(n^3)$, chạy vài ngày hoặc hết RAM | `LinearSVC`, `SGDClassifier(loss='hinge')`, hoặc `Nystroem` + linear |
| 7 | Bỏ qua mất cân bằng lớp | SVM tối thiểu tổng hinge loss, nên siêu phẳng bị đẩy về phía lớp ít, recall lớp hiếm rất thấp | `class_weight='balanced'` (đặt $C_k \propto n/(K \cdot n_k)$) và đánh giá bằng F1/recall thay vì accuracy |
| 8 | Đọc `coef_` khi dùng kernel phi tuyến | `coef_` chỉ tồn tại với `kernel='linear'`; với RBF, $w$ sống trong không gian vô hạn chiều nên không viết ra được | Dùng permutation importance hoặc SHAP nếu cần diễn giải |
| 9 | Tin train accuracy 100% | Với $\gamma$ đủ lớn, RBF luôn đạt 100% train accuracy trên mọi dataset (kể cả nhãn ngẫu nhiên) | Luôn nhìn CV score và tỷ lệ số SV / số mẫu |


## Tổng kết

1. SVM tìm siêu phẳng có **margin tối đa**.
2. **Tham số C**: kiểm soát đánh đổi giữa margin lớn (C nhỏ) và phạt vi phạm (C lớn).
3. **Kernel trick**: cho phép học ranh giới phi tuyến mà không cần dựng tường minh không gian cao chiều.
4. RBF kernel là default tốt; chỉ cần tune $C$ và $\gamma$.
5. Phải scale feature, vì SVM rất nhạy với scale.
6. Nghiệm SVM chỉ phụ thuộc vào các support vector, nên mô hình thưa một cách tự nhiên.

## Khi nào dùng SVM?
- Dataset vừa và nhỏ (vài nghìn đến vài chục nghìn).
- Dữ liệu có chiều cao nhưng vẫn ít hơn số mẫu.
- Cần ranh giới quyết định rõ ràng.

## Khi nào tránh SVM?
- Dataset rất lớn (trên 100k mẫu), train chậm.
- Cần predict probability đáng tin (SVM không cho probability tự nhiên; `probability=True` chậm và không phải lúc nào cũng tốt).

# BÀI TẬP VỀ NHÀ

## Bài 1: Vẽ ảnh hưởng của gamma trong RBF
Trên `make_moons`, train SVM RBF với `gamma ∈ {0.1, 1, 10, 100}`, C cố định = 1. Vẽ 4 decision boundary. Quan sát:
- gamma nhỏ cho boundary mượt (underfit).
- gamma lớn cho boundary bám theo từng điểm (overfit).

## Bài 2: SVM trên drug200
Áp dụng SVM lên drug200 (như bài Decision Tree). Sweep `kernel ∈ {linear, rbf, poly}`. So sánh với Random Forest. Cái nào tốt hơn? Vì sao?

## Bài 3: Hard-margin từ scratch (chỉ làm khi đã có cvxopt)
Cài `pip install cvxopt`. Implement hard-margin SVM bằng quadratic programming:
1. Thiết lập ma trận P, q, G, h, A, b cho dual problem.
2. Gọi `cvxopt.solvers.qp()`.
3. Lấy support vectors (có $\lambda_i > 10^{-5}$).
4. Tính w, b. Vẽ ranh giới.

*Gợi ý:* $P_{ij} = y_i y_j x_i^T x_j$, $q = -\mathbf{1}$, ràng buộc $\lambda_i \ge 0$ là $G = -I$, $h = 0$. Ràng buộc $\sum \lambda_i y_i = 0$ là $A = y^T$, $b = 0$.

## Bài 4: Imbalanced classes
Sinh dữ liệu mất cân bằng: 950 mẫu lớp 0, 50 mẫu lớp 1. Train SVM. Quan sát: accuracy cao nhưng recall lớp 1 thấp. Thử `class_weight='balanced'`: cải thiện thế nào? Vì sao?

## Bài 5: Probability calibration
SVM `decision_function` cho điểm số, không phải xác suất. Dùng `CalibratedClassifierCV` (Platt scaling) để có xác suất tin cậy. Trên Iris, xét bài toán nhị phân versicolor so với hai loài còn lại (setosa tách được hoàn toàn nên không thú vị). Vẽ ROC curve và calibration curve, so sánh Brier score trước và sau calibration. AUC có thay đổi không? Vì sao (gợi ý: Platt scaling là một phép biến đổi đơn điệu của điểm số)?

*Gợi ý:* `from sklearn.calibration import CalibratedClassifierCV`.